In [ ]:
# Phase 6 screen, fold 0. Runs on prep v2 artifacts (130mm physical-scale crop
# at 336px, six plane/weighting slots with a presence mask, contiguous 3-slice
# groups) and the report-hash folds v3.
#
# The first config is NOT a screen cell: it is Step 3's re-baseline, the row every
# later delta is measured against. It cannot be paired against Phase 5's 0.8484
# because the two do not see the same pixels -- singleton evenly-spaced slices on
# a 256px letterbox versus 3-slice groups at 130mm. Expect within ~0.01 with the
# per-label profile intact; a large drop means the prep is wrong.
import glob, os, shutil, sys, time

GIT_SHA = '801df73-wip'

SRC = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)[0]
COMP_DIR = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)[0]

PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')

import inspect
import torch

assert torch.cuda.is_available()
major, minor = torch.cuda.get_device_capability(0)
print(f'GPU: {torch.cuda.get_device_name(0)}, sm_{major}{minor}')
assert (major, minor) >= (7, 0), 'need T4, not P100'
device = 'cuda'

# A stale rsna-knee-src version cost 7.5 minutes of a Phase 5 screen before the
# run noticed it was training the wrong code (NOTES 2026-09-05). These asserts
# fail in the first cell instead.
from knee.dataset import PreppedSlotDataset
from knee.model import KneeModel
from knee.reports import report_group_key
from knee.train import differential_param_groups, make_folds, train_one_epoch
# A stale src dataset has now cost three runs. The old guard listed the previous
# change by hand, so it only ever caught yesterday's mistake -- this one is
# derived from what the configs below actually ask for, so a config needing new
# code cannot silently outrun the dataset version. Tuples, not sets: this cell is
# an f-string and braces here would be interpolated.
REQUIRED = [
    ('KneeModel.__init__', KneeModel.__init__, ('head', 'backbone_kwargs')),
    ('KneeModel.forward', KneeModel.forward, ('mask',)),
    ('make_folds', make_folds, ('groups',)),
    ('differential_param_groups', differential_param_groups, ('backbone_lr', 'head_lr')),
]
for label, fn, needed in REQUIRED:
    have = set(inspect.signature(fn).parameters)
    missing = [n for n in needed if n not in have]
    assert not missing, (label + ' is missing ' + repr(missing) +
                         ' -- rsna-knee-src is stale. Re-version it before pushing.')
assert hasattr(KneeModel, 'attention_weights'), 'rsna-knee-src predates the c2 head'
print('src carries every Phase 6 change this run depends on | GIT_SHA', GIT_SHA)

In [ ]:
# GATE -- the corpus prep census, read from the shard manifests rather than
# downloaded: 9.5GB of artifacts do not need to leave Kaggle to be checked.
import numpy as np
import pandas as pd

PREPPED_DIRS = sorted(glob.glob('/kaggle/input/**/prepped', recursive=True))
uid_to_npz = {}
for d in PREPPED_DIRS:
    for f in os.listdir(d):
        if f.endswith('.npz'):
            uid_to_npz[f[:-4]] = os.path.join(d, f)

manifest = pd.concat([pd.read_csv(f) for f in
                      sorted(glob.glob('/kaggle/input/**/prep_manifest_shard*.csv', recursive=True))])
failed = pd.concat([pd.read_csv(f) for f in
                    sorted(glob.glob('/kaggle/input/**/prep_failed_shard*.csv', recursive=True))])
series_meta = pd.concat([pd.read_csv(f) for f in
                         sorted(glob.glob('/kaggle/input/**/prep_series_meta_shard*.csv', recursive=True))])

# One root by symlink, as Phase 5 did: the loader takes a directory and the
# artifacts arrive in four mounted shards. Symlinks cost no space.
NPZ_ROOT = '/kaggle/working/prepped_all'
os.makedirs(NPZ_ROOT, exist_ok=True)
for uid, p in uid_to_npz.items():
    dst = os.path.join(NPZ_ROOT, f'{uid}.npz')
    if not os.path.exists(dst):
        os.symlink(p, dst)

print(f'{len(PREPPED_DIRS)} shards, {len(uid_to_npz)} artifacts, '
      f'{len(manifest)} manifest rows, {len(failed)} failed studies')
assert len(uid_to_npz) == len(manifest), 'artifact count and manifest disagree'
assert len(uid_to_npz) + len(failed) == 4407, 'shards do not cover the corpus'

mm = series_meta['mm_per_px'].unique()
assert len(mm) == 1, f'physical scale is not constant: {mm}'
print(f'PASS: {len(series_meta)} stored series, {series_meta.PixelSpacing.nunique()} distinct '
      f'PixelSpacing values, all at {mm[0]:.5f} mm/px')

from knee.dicom import SLOTS
slot_names = [n for n, _, _ in SLOTS]
fill = manifest.slots_filled.fillna('').str.get_dummies(sep='|')
print(f'\nmean slots/study {manifest.n_slots_filled.mean():.2f} of 6')
for name in slot_names:
    n = int(fill[name].sum()) if name in fill else 0
    print(f'  {name:12s} {n:5d}  {n / len(manifest):6.1%}')
print(f'\ndecode failures: {int(manifest.n_decode_failures.sum())}')
print(f'laterality routes: {dict(manifest.route.value_counts())}')

In [ ]:
# Folds v3 and labels. Folds are regenerated here rather than uploaded, and
# pinned by hash: the experiment discipline requires one frozen fold assignment
# across every Phase 6 run, and a silent drift would make every paired delta
# meaningless without failing anything.
import hashlib

from knee.infer import LABEL_COLUMNS
from knee.metrics import macro_auc, paired_macro_auc_delta, per_label_auc
from knee.train import Timer, evaluate, log_experiment, train_val_split
from torch.utils.data import DataLoader

train_df = pd.read_csv(f'{COMP_DIR}/train.csv')
all_uids = sorted(train_df['StudyInstanceUID'].astype(str))
assert len(all_uids) == 4407

groups = {u: report_group_key(r)
          for u, r in zip(train_df['StudyInstanceUID'].astype(str), train_df['Report'])}
folds = make_folds(all_uids, n_folds=5, seed=0, groups=groups)
FOLDS_PRIMARY_V3_SHA256 = 'f1d6ba7c341f8d2e82241cb4ddd5ca5131b0c6d2033eb6d2e43bcb597cf12868'
assert hashlib.sha256(
    '\n'.join(f'{u},{folds[u]}' for u in sorted(folds)).encode()
).hexdigest() == FOLDS_PRIMARY_V3_SHA256, 'fold assignment drifted from folds_primary_v3.csv'
n_straddle = sum(1 for _, g in pd.DataFrame(
    {'g': [groups[u] for u in all_uids], 'f': [folds[u] for u in all_uids]}
).groupby('g') if g.f.nunique() > 1)
print(f'folds v3 verified; {n_straddle} report groups straddle folds (v2 had 47)')

gold_cols = [c for c in train_df.columns if c not in ('StudyInstanceUID', 'Report')]
gold_df = train_df[train_df[gold_cols].notna().any(axis=1)]
holdout = frozenset(gold_df['StudyInstanceUID'].astype(str))
print(f'{len(holdout)} gold studies held out of both sides of every split')

# Step 3 re-baselines on the same label source Phase 5 trained on; swapping the
# label target is c0, a separate cell, so the two changes stay attributable.
pseudo = pd.read_csv(glob.glob('/kaggle/input/**/pseudo_labels_qwen3_4b.csv', recursive=True)[0])
labels_all = pseudo[['StudyInstanceUID'] + [f'score_{l}' for l in LABEL_COLUMNS]]
labels_all.columns = ['StudyInstanceUID'] + LABEL_COLUMNS

# Two frames, as Phase 5 used: soft scores are the training target, but AUC is
# scored against a binarized one -- roc_auc_score rejects a continuous y_true,
# and the soft target is a confidence, not a label.
labels_eval = labels_all.copy()
labels_eval[LABEL_COLUMNS] = (labels_all[LABEL_COLUMNS].to_numpy(dtype=float) >= 0.5).astype(float)
print(f'{len(labels_all)} pseudo-labelled studies (soft for training, binarized for scoring)')

# c0's alternative target. steven_v4 is the best single public source on the 58
# gold studies (0.8927 vs our 0.8616, paired delta +0.031 [+0.004, +0.058]).
#
# NOT the 5-source rank blend, which scored higher on gold (0.8976): percentile
# ranks are uniform by construction, so every label's target mean is exactly
# 0.500 and all prevalence information is destroyed -- Fracture would be trained
# at 0.5 against a real rate of 0.014. AUC never notices, because AUC reads
# order only. That is exactly why rank-averaging is right for combining
# predictions at submission time and wrong for building a target.
LABEL_SOURCES = {'ours_qwen3_4b': (labels_all, labels_eval)}
steven = glob.glob('/kaggle/input/**/llm_labels_v4_blend.csv', recursive=True)
if steven:
    alt = pd.read_csv(steven[0])
    alt['StudyInstanceUID'] = alt['StudyInstanceUID'].astype(str)
    alt = alt[['StudyInstanceUID'] + LABEL_COLUMNS]
    alt_eval = alt.copy()
    alt_eval[LABEL_COLUMNS] = (alt[LABEL_COLUMNS].to_numpy(dtype=float) >= 0.5).astype(float)
    LABEL_SOURCES['steven_v4'] = (alt, alt_eval)
    rate = alt[LABEL_COLUMNS].mean()
    print(f'steven_v4 loaded: {len(alt)} studies, positive rate '
          f'{rate.min():.3f}-{rate.max():.3f} (ours: '
          f'{labels_all[LABEL_COLUMNS].mean().min():.3f}-'
          f'{labels_all[LABEL_COLUMNS].mean().max():.3f})')
print('label sources available:', list(LABEL_SOURCES))

# The arbiter for c0. Scoring each model against its OWN label source would be
# circular -- that measures how learnable a label set is, not which one trains a
# better model, and each source would be graded by its own marker. The 58 gold
# studies are rubric-graded from images, excluded from every training split, and
# identical for both arms, so they are the only unbiased comparison available
# when the target itself is the variable. n=58 is thin (MCL has 9 positives) and
# the per-label numbers are directional only; the macro is the read.
gold_labels = gold_df[['StudyInstanceUID'] + LABEL_COLUMNS].copy()
gold_labels['StudyInstanceUID'] = gold_labels['StudyInstanceUID'].astype(str)
gold_uids = [u for u in gold_labels['StudyInstanceUID'] if u in uid_to_npz]
print(f'gold arbiter: {len(gold_uids)} of 58 studies have a prepped artifact')

In [ ]:
# The runs. One config at a time, each logged; a time guard stops the session
# before Kaggle does rather than losing every result to a 12h kill.
#
# No `sides` override here, unlike Phase 5. That override existed because the v1
# artifacts were prepped before the geometry laterality route and carried
# side=None for ~49% of studies. prep_slots calls census_study_laterality, so a
# v2 artifact already carries the geometry-resolved side (pilot routes:
# Laterality 34, geometry 27, SeriesDescription 2, unresolved 2) -- re-applying
# an external map would be a second source of truth for the same fact.

EXPERIMENTS_CSV = '/kaggle/working/experiments.csv'
_HEADER = ('date,git_sha,config_hash,hypothesis,fold_set,seed,acl_auc,mcl_auc,'
           'medial_meniscus_auc,lateral_meniscus_auc,medial_oa_auc,lateral_oa_auc,'
           'pf_oa_auc,effusion_auc,synovitis_auc,bakers_auc,contusion_auc,fracture_auc,'
           'macro_auc,paired_delta,train_minutes,inference_seconds,promoted')
with open(EXPERIMENTS_CSV, 'w') as f:
    print(_HEADER, file=f)

# Both arms run here rather than pairing c0 against the v1 screen's row: that run
# lost its checkpoint to the del-ordering bug, so it has no gold predictions, and
# a gold comparison needs both arms scored in the same session anyway.
# c0 promoted (gold +0.0339 [+0.0067, +0.0625]), so steven_v4 is the target from
# here on and c0 is the row later cells pair against. Both arms of a cell run in
# one session whenever the comparison needs the same session -- fold-0 run-to-run
# noise is ~0.0065, so a delta under ~0.007 is not a result.
# c3 is sized to pair against c0's saved arrays -- same 2 slots x 5 groups, same
# target, same mean-pool head -- so the backbone is the only thing that moves and
# no control has to be re-run. It cannot take c2's six slots: vit_small_patch14 is
# ~15x efficientnet_b0's FLOPs, which puts 6x5 at ~13h, past the session cap.
# The differential learning rate is not a separable second variable: a pretrained
# ViT trained at OneCycle 3e-4 would simply be destroyed, so "use this backbone"
# and "run its trunk slower than its head" are one change.
CONFIGS = [
    # name, source, slots, n_groups, out_size, epochs, batch, head, backbone, lrs
    ('c5_dinov2_6slot_attn', 'steven_v4', slot_names, 5, 224, 8, 4, 'slot_attention',
     'vit_small_patch14_dinov2.lvd142m', (8e-6, 1e-3)),
]
HYPOTHESIS = {
    'rebaseline_2slot_224':
        'Step 3 re-baseline: the Phase 5 winner re-run on prep v2 (130mm crop, '
        '3-slice groups) and folds v3, reading 2 of 6 slots at 224. Establishes '
        'the row every Phase 6 screen delta is measured against; NOT paired '
        'against Phase 5, which saw different pixels.',
    'c0_steven_v4':
        'c0 (re-run as the pairing partner): steven_v4 as the training target, '
        'promoted on gold transfer +0.0339 [+0.0067, +0.0625].',
    'c5_dinov2_6slot_attn':
        'c5: the stack -- DINOv2 trunk, six slots, per-diagnosis attention. Pairs '
        'against c2 with only the backbone moving. c3 showed the backbone is null '
        'at two slots with a mean pool, but it was underfit and its per-label '
        'strengths are complementary; screen B is the precedent for a change that '
        'reads as nothing alone and pays in combination.',
    'c3_dinov2_224':
        'c3: swap efficientnet_b0 for DINOv2-small (self-supervised, 21.6M params) '
        'with the trunk at 8e-6 and the head at 1e-3. Held at c0 shape (2 slots x '
        '5 groups, mean pool) so it pairs against c0 directly and the backbone is '
        'the only thing that moves.',
    'c2_6slot_attn':
        'c2: per-diagnosis masked attention over the six slots, replacing the '
        'mean pool. Pairs against c1 (same slots, same target, mean pool) on the '
        'gold set. The claim is that each diagnosis can read its own plane; the '
        'attention weights are saved so that claim is checkable against the '
        'anatomy rather than assumed from a macro delta.',
    'c1_6slot':
        'c1: all six plane/weighting slots with the presence mask instead of two, '
        'target and schedule held at c0. Does the extra input carry signal under a '
        'mean-pool head at all -- AX_STRUCT is present for only 19.4% of studies, '
        'so part of the answer is how much of the new input actually exists.',
}
FOLD = 0
TIME_BUDGET_S = 10.5 * 3600
t_start = time.time()

def loader(uids, labels_df, cfg_slots, n_groups, out_size, batch,
           shuffle=False, augment=False):
    ds = PreppedSlotDataset(uids, NPZ_ROOT, labels_df=labels_df, n_groups=n_groups,
                            out_size=out_size, slots=cfg_slots, augment=augment)
    return DataLoader(ds, batch_size=batch, shuffle=shuffle, num_workers=2)

results = []
for (name, source, cfg_slots, n_groups, out_size, epochs, batch, head,
     backbone, lrs) in CONFIGS:
    train_labels, eval_labels = LABEL_SOURCES[source]
    if time.time() - t_start > TIME_BUDGET_S:
        print(f'time budget spent, skipping {name}')
        break
    tr_uids, va_uids = train_val_split(folds, val_fold=FOLD, exclude_uids=holdout)
    tr_uids = [u for u in tr_uids if u in uid_to_npz]
    va_uids = [u for u in va_uids if u in uid_to_npz]
    print(f'\n=== {name}: {len(tr_uids)} train / {len(va_uids)} val, '
          f'{len(cfg_slots)} slots x {n_groups} groups at {out_size}px, {epochs} epochs')

    train_loader = loader(tr_uids, train_labels, cfg_slots, n_groups, out_size, batch,
                          shuffle=True, augment=True)
    val_loader = loader(va_uids, eval_labels, cfg_slots, n_groups, out_size, batch)
    extra = {'img_size': out_size} if backbone.startswith('vit_') else {}
    model = KneeModel(backbone_name=backbone, num_labels=12, pretrained=True,
                      head=head, **extra).to(device)
    if lrs is None:
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
        max_lr = 3e-4
    else:
        backbone_lr, head_lr = lrs
        optimizer = torch.optim.AdamW(
            differential_param_groups(model, backbone_lr, head_lr), weight_decay=0.02)
        # OneCycle takes one max_lr per group, in the order the groups were built
        max_lr = [backbone_lr * 3, head_lr * 3]
    scaler = torch.amp.GradScaler('cuda')
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=max_lr, epochs=epochs, steps_per_epoch=len(train_loader))

    t0 = time.time()
    for epoch in range(epochs):
        loss = train_one_epoch(model, train_loader, optimizer, device=device,
                               scaler=scaler, scheduler=scheduler)
        print(f'  epoch {epoch + 1}/{epochs} loss {loss:.4f} '
              f'({(time.time() - t0) / 60:.1f} min)')
    minutes = (time.time() - t0) / 60

    y_true, y_pred = evaluate(model, val_loader, device=device)
    aucs = per_label_auc(y_true, y_pred)
    macro = macro_auc(y_true, y_pred)

    # Tier 2: the same 58 rubric-graded studies for every arm, whatever it trained on
    gold_loader = loader(gold_uids, gold_labels, cfg_slots, n_groups, out_size, batch)
    g_true, g_pred = evaluate(model, gold_loader, device=device)
    gold_macro = macro_auc(g_true, g_pred)
    np.save(f'/kaggle/working/gold_pred_{name}.npy', g_pred)
    np.save(f'/kaggle/working/gold_true_{name}.npy', g_true)
    if head == 'slot_attention':
        # which plane each diagnosis actually reads, averaged over the gold
        # studies. A macro delta says whether c2 helped; this says whether it
        # helped for the reason claimed.
        model.eval()
        chunks = []
        with torch.no_grad():
            for images, m, _, _ in gold_loader:
                chunks.append(model.attention_weights(images.to(device),
                                                      m.to(device)).cpu().numpy())
        attn = np.concatenate(chunks)
        np.save(f'/kaggle/working/attn_{name}.npy', attn)
        mean_attn = attn.mean(axis=0)
        print('    attention over slots, averaged across gold studies:')
        print('      ' + ' '.join(f'{s[:9]:>9s}' for s in cfg_slots))
        for li, label in enumerate(LABEL_COLUMNS):
            print(f'      {label:20s} ' +
                  ' '.join(f'{mean_attn[si, li]:9.3f}' for si in range(len(cfg_slots))))

    print(f'  {name}: val macro {macro:.4f} (own target), '
          f'GOLD macro {gold_macro:.4f} in {minutes:.1f} min')
    for label, auc in zip(LABEL_COLUMNS, aucs):
        print(f'    {label:20s} {auc:.4f}')
    results.append({'config': name, 'source': source, 'macro_auc': macro,
                    'gold_macro': gold_macro, 'minutes': minutes,
                    **dict(zip(LABEL_COLUMNS, aucs))})
    # "If a run isn't in here it didn't happen." paired_delta is nan for the
    # re-baseline on purpose: it has nothing to pair against -- Phase 5 saw
    # different pixels, and a 0.0 there would read as a measured null.
    log_experiment(EXPERIMENTS_CSV, git_sha=GIT_SHA, config_hash=name,
                   hypothesis=HYPOTHESIS[name], fold_set=f'primary_v3_fold{FOLD}',
                   seed=0, per_label_auc={l: float(a) for l, a in zip(LABEL_COLUMNS, aucs)},
                   macro_auc=float(macro), paired_delta=float('nan'),
                   train_minutes=minutes, inference_seconds=float('nan'), promoted=False)
    np.save(f'/kaggle/working/oof_{name}.npy', y_pred)
    np.save(f'/kaggle/working/y_{name}.npy', y_true)
    torch.save(model.state_dict(), f'/kaggle/working/{name}.pt')
    # freed only after every consumer of `model` has run -- an earlier del cost
    # run 1 its checkpoint (the arrays had already been saved, so only the
    # weights were lost)
    del model, optimizer, scaler, scheduler
    torch.cuda.empty_cache()

In [ ]:
# Results table. Phase 5's fold-0 number is printed alongside for orientation
# only -- it is NOT a paired comparison and no promotion decision reads it.
PHASE5_FOLD0_MACRO = 0.8484

table = pd.DataFrame(results)
table.to_csv('/kaggle/working/phase6_screen_fold0.csv', index=False)
print(table[['config', 'source', 'macro_auc', 'gold_macro', 'minutes']].to_string(index=False))

# c0's decision, if both arms ran. Val macro is NOT comparable across arms --
# each is scored against its own label source. Gold is.
if len(results) == 2:  # only when a session runs both arms of a comparison
    a, b = results
    delta, lo, hi = paired_macro_auc_delta(
        np.load(f"/kaggle/working/gold_true_{a['config']}.npy"),
        np.load(f"/kaggle/working/gold_pred_{a['config']}.npy"),
        np.load(f"/kaggle/working/gold_pred_{b['config']}.npy"))
    print()
    print(f"gold transfer, {b['config']} minus {a['config']}: "
          f"{delta:+.4f} [{lo:+.4f}, {hi:+.4f}]")
    print('  CI excludes zero -- the label swap is real'
          if lo > 0 or hi < 0 else
          '  CI spans zero -- n=58 cannot separate these; keep the incumbent target')
print(f'\nPhase 5 b4 on fold 0 (different pixels, unpaired): {PHASE5_FOLD0_MACRO:.4f}')
for row in results:
    delta = row['macro_auc'] - PHASE5_FOLD0_MACRO
    verdict = ('within the expected band' if abs(delta) <= 0.01 else
               'OUTSIDE the band -- check the prep before screening further')
    print(f"  {row['config']}: {row['macro_auc']:.4f} ({delta:+.4f}) -- {verdict}")